# AntiFraud Agent - Analysis Notebook

This notebook provides interactive analysis capabilities for fraud detection findings.

## Features
- Load and explore fraud findings
- Visualize threat trends
- IOC analysis and enrichment
- Custom ML model training
- YARA rule generation
- Report generation

In [ ]:
# Setup
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json

# AntiFraud modules
from agents.fraud_agent import FraudDetectionAgent
from llm.agent_llm import AgentLLM
from cti.manager import CTIManager
from ml.fraud_classifier import FraudClassifier
from exporters.yara_generator import YARAGenerator
from config.config_manager import ConfigManager

# Visualization setup
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Environment setup complete")

## 1. Load Configuration and Initialize Components

In [ ]:
# Load configuration
config_manager = ConfigManager()
config = config_manager.load_config()

print("Configuration loaded:")
print(f"- LLM Backend: {config.get('llm_backend')}")
print(f"- URLHaus: {config.get('urlhaus_enabled')}")
print(f"- VirusTotal: {config.get('virustotal_enabled')}")
print(f"- Trend Vision One: {config.get('trend_enabled')}")

## 2. Load Fraud Findings Data

In [ ]:
# Load recent findings from data directory
import glob
import os

findings_files = glob.glob('../data/findings/*.json')
findings_files.sort(reverse=True)  # Most recent first

all_findings = []

for file_path in findings_files[:10]:  # Load last 10 files
    with open(file_path, 'r') as f:
        data = json.load(f)
        findings = data.get('findings', [])
        all_findings.extend(findings)

print(f"Loaded {len(all_findings)} findings from {len(findings_files[:10])} files")

# Convert to DataFrame
if all_findings:
    df = pd.DataFrame(all_findings)
    print("\nDataFrame shape:", df.shape)
    print("\nColumns:", list(df.columns))
    display(df.head())
else:
    print("\nNo findings found. Run a workflow first to generate data.")
    df = pd.DataFrame()

## 3. Fraud Type Distribution

In [ ]:
if not df.empty and 'fraud_type' in df.columns:
    # Fraud type distribution
    fraud_counts = df['fraud_type'].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Bar chart
    fraud_counts.plot(kind='bar', ax=ax1, color='coral')
    ax1.set_title('Fraud Type Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Fraud Type')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # Pie chart
    fraud_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%')
    ax2.set_title('Fraud Type Proportion', fontsize=14, fontweight='bold')
    ax2.set_ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    print("\nFraud Type Statistics:")
    print(fraud_counts)
else:
    print("No fraud_type data available")

## 4. Confidence Score Analysis

In [ ]:
if not df.empty and 'confidence' in df.columns:
    # Convert confidence to numeric
    df['confidence_numeric'] = pd.to_numeric(df['confidence'], errors='coerce')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    df['confidence_numeric'].hist(bins=20, ax=ax1, color='skyblue', edgecolor='black')
    ax1.set_title('Confidence Score Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Confidence (%)')
    ax1.set_ylabel('Frequency')
    ax1.axvline(df['confidence_numeric'].mean(), color='red', linestyle='--', label='Mean')
    ax1.legend()
    
    # Box plot by fraud type
    if 'fraud_type' in df.columns:
        df.boxplot(column='confidence_numeric', by='fraud_type', ax=ax2)
        ax2.set_title('Confidence by Fraud Type', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Fraud Type')
        ax2.set_ylabel('Confidence (%)')
        plt.suptitle('')  # Remove auto-title
    
    plt.tight_layout()
    plt.show()
    
    print("\nConfidence Statistics:")
    print(df['confidence_numeric'].describe())
else:
    print("No confidence data available")

## 5. Timeline Analysis

In [ ]:
if not df.empty and 'timestamp' in df.columns:
    # Convert timestamp to datetime
    df['datetime'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['date'] = df['datetime'].dt.date
    
    # Daily findings
    daily_counts = df.groupby('date').size()
    
    plt.figure(figsize=(12, 5))
    daily_counts.plot(kind='line', marker='o', color='darkgreen')
    plt.title('Daily Fraud Findings Trend', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Number of Findings')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("\nDaily Statistics:")
    print(daily_counts.describe())
else:
    print("No timestamp data available")

## 6. IOC Analysis

In [ ]:
# Analyze IOCs from findings
all_iocs = []

for finding in all_findings:
    iocs = finding.get('iocs', [])
    if isinstance(iocs, list):
        all_iocs.extend(iocs)

print(f"Total IOCs extracted: {len(all_iocs)}")
print(f"Unique IOCs: {len(set(all_iocs))}")

if all_iocs:
    # IOC type distribution
    ioc_types = {'urls': 0, 'domains': 0, 'ips': 0, 'hashes': 0, 'other': 0}
    
    import re
    for ioc in all_iocs:
        if ioc.startswith('http'):
            ioc_types['urls'] += 1
        elif re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', ioc):
            ioc_types['ips'] += 1
        elif len(ioc) in [32, 40, 64]:
            ioc_types['hashes'] += 1
        elif '.' in ioc:
            ioc_types['domains'] += 1
        else:
            ioc_types['other'] += 1
    
    # Plot
    plt.figure(figsize=(10, 5))
    pd.Series(ioc_types).plot(kind='bar', color='orange')
    plt.title('IOC Type Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('IOC Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("\nIOC Type Breakdown:")
    for ioc_type, count in ioc_types.items():
        print(f"  {ioc_type}: {count}")

## 7. Train Custom ML Model

In [ ]:
# Train fraud classifier on collected data
if not df.empty and 'fraud_type' in df.columns:
    # Prepare training data
    training_data = []
    labels = []
    
    for _, row in df.iterrows():
        if pd.notna(row.get('fraud_type')):
            item = {
                'text': str(row.get('explanation', '')),
                'url': str(row.get('source_url', '')),
                'suspicious_patterns': [],
                'iocs': []
            }
            training_data.append(item)
            labels.append(row['fraud_type'])
    
    if len(training_data) >= 10:  # Minimum samples
        # Initialize classifier
        classifier = FraudClassifier(model_type='random_forest')
        
        # Train
        print("Training model...")
        results = classifier.train(training_data, labels, test_size=0.2)
        
        print("\n=== Training Results ===")
        print(f"Accuracy: {results['accuracy']:.2%}")
        print(f"Training Samples: {results['training_samples']}")
        print(f"Test Samples: {results['test_samples']}")
        
        # Save model
        classifier.save_model('../models/fraud_classifier.pkl')
        print("\n✓ Model saved to models/fraud_classifier.pkl")
        
        # Feature importance
        importance = classifier.get_feature_importance()
        if importance:
            print("\nTop 10 Important Features:")
            for feature, score in importance[:10]:
                print(f"  {feature}: {score:.4f}")
    else:
        print(f"Insufficient training data ({len(training_data)} samples). Need at least 10.")
else:
    print("No training data available")

## 8. Generate YARA Rules

In [ ]:
# Generate YARA rules from high-confidence findings
if all_findings:
    yara_gen = YARAGenerator()
    
    # Generate rules for findings with confidence >= 70
    rules = yara_gen.generate_rules_from_findings(all_findings, min_confidence=70)
    
    print(f"Generated {len(rules)} YARA rules")
    
    if rules:
        # Save rules
        yara_gen.save_rules(rules, '../data/yara_rules.yar')
        print("\n✓ YARA rules saved to data/yara_rules.yar")
        
        # Display first rule
        print("\n=== Sample YARA Rule ===")
        print(rules[0][:500] + "..." if len(rules[0]) > 500 else rules[0])
else:
    print("No findings available for YARA rule generation")

## 9. Manual URL Analysis

In [ ]:
# Analyze a URL on-demand
import asyncio

async def analyze_url(url):
    # Initialize components
    llm = AgentLLM(backend=config.get('llm_backend', 'local'), 
                   model_path=config.get('model_path'))
    cti_manager = CTIManager(
        virustotal_api_key=config.get('virustotal_api_key')
    )
    agent = FraudDetectionAgent(config, llm, cti_manager)
    
    # Analyze
    result = await agent.analyze_url(url)
    return result

# Example usage (uncomment to run)
# url_to_analyze = "https://example.com"
# result = await analyze_url(url_to_analyze)
# print(json.dumps(result, indent=2))

## 10. Export Report

In [ ]:
# Generate summary report
if not df.empty:
    report = f"""
# Fraud Analysis Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Summary
- Total Findings: {len(df)}
- Date Range: {df['date'].min()} to {df['date'].max()}
- Average Confidence: {df['confidence_numeric'].mean():.1f}%

## Fraud Type Breakdown
{df['fraud_type'].value_counts().to_string()}

## High Confidence Findings
Findings with confidence >= 80%: {len(df[df['confidence_numeric'] >= 80])}

## Top Source URLs
{df['source_url'].value_counts().head(10).to_string()}

## IOC Summary
- Total IOCs: {len(all_iocs)}
- Unique IOCs: {len(set(all_iocs))}
    """
    
    # Save report
    report_path = f"../data/analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    with open(report_path, 'w') as f:
        f.write(report)
    
    print(report)
    print(f"\n✓ Report saved to {report_path}")
else:
    print("No data available for report generation")